# TabGAN — Synthetic Tabular Data Generation Examples

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Diyago/Tabular-data-generation/blob/master/examples/tabgan_examples.ipynb)
[![GitHub](https://img.shields.io/github/stars/Diyago/Tabular-data-generation?style=social)](https://github.com/Diyago/Tabular-data-generation)
[![PyPI](https://img.shields.io/pypi/v/tabgan)](https://pypi.org/project/tabgan/)
[![HF Space](https://img.shields.io/badge/%F0%9F%A4%97%20Spaces-TabGAN%20Demo-blue)](https://huggingface.co/spaces/InsafQ/TabGAN)

This notebook demonstrates all major features of **TabGAN**. Most examples run on CPU; LLM-based generation benefits from GPU.

> **Tip:** For GPU runtime, go to *Runtime > Change runtime type > T4 GPU*.

## 0. Installation

In [ ]:
!pip install -q tabgan

## 1. Quick Start

In [ ]:
import pandas as pd
import numpy as np
from tabgan.sampler import GANGenerator

train = pd.DataFrame(np.random.randint(-10, 150, size=(150, 4)), columns=list("ABCD"))
target = pd.DataFrame(np.random.randint(0, 2, size=(150, 1)), columns=list("Y"))
test = pd.DataFrame(np.random.randint(0, 100, size=(100, 4)), columns=list("ABCD"))

new_train, new_target = GANGenerator(
    gen_params={"batch_size": 500, "epochs": 10}
).generate_data_pipe(train, target, test)

print(f"Original: {train.shape}, Synthetic: {new_train.shape}")
new_train.head()

## 2. All Generators Comparison

In [ ]:
from tabgan.sampler import OriginalGenerator, GANGenerator, ForestDiffusionGenerator

train = pd.DataFrame(np.random.randint(-10, 150, size=(150, 4)), columns=list("ABCD"))
target = pd.DataFrame(np.random.randint(0, 2, size=(150, 1)), columns=list("Y"))
test = pd.DataFrame(np.random.randint(0, 100, size=(100, 4)), columns=list("ABCD"))

# Original (random sampling baseline)
new_train1, new_target1 = OriginalGenerator().generate_data_pipe(train, target, test)
print(f"OriginalGenerator: {new_train1.shape}")

# GAN (CTGAN)
new_train2, new_target2 = GANGenerator(
    gen_params={"batch_size": 500, "epochs": 10, "patience": 5}
).generate_data_pipe(train, target, test)
print(f"GANGenerator:      {new_train2.shape}")

# Forest Diffusion
new_train3, new_target3 = ForestDiffusionGenerator().generate_data_pipe(train, target, test)
print(f"ForestDiffusion:   {new_train3.shape}")

### LLM Generator (optional, requires GPU)

Uncomment and run if you have a GPU runtime. Downloads `distilgpt2` (~250MB).

In [ ]:
# from tabgan.sampler import LLMGenerator
#
# new_train4, new_target4 = LLMGenerator(
#     gen_params={"batch_size": 32, "epochs": 2, "llm": "distilgpt2", "max_length": 500}
# ).generate_data_pipe(train, target, test)
# print(f"LLMGenerator:      {new_train4.shape}")

## 3. Full Parameter Example

In [ ]:
new_train, new_target = GANGenerator(
    gen_x_times=1.1,
    cat_cols=None,
    bot_filter_quantile=0.001,
    top_filter_quantile=0.999,
    is_post_process=True,
    adversarial_model_params={
        "metrics": "AUC", "max_depth": 2, "max_bin": 100,
        "learning_rate": 0.02, "random_state": 42, "n_estimators": 100,
    },
    pregeneration_frac=2,
    only_generated_data=False,
    gen_params={"batch_size": 500, "patience": 25, "epochs": 10},  # reduced for demo
).generate_data_pipe(
    train, target, test,
    deep_copy=True,
    only_adversarial=False,
    use_adversarial=True,
)

print(f"Synthetic: {new_train.shape}")
new_train.head()

## 4. LLM Conditional Text Generation (GPU recommended)

In [ ]:
# Uncomment to run (requires GPU runtime for reasonable speed)

# from tabgan.sampler import LLMGenerator
#
# train_llm = pd.DataFrame({
#     "Name": ["Anna", "Maria", "Ivan", "Sergey", "Olga", "Boris"],
#     "Gender": ["F", "F", "M", "M", "F", "M"],
#     "Age": [25, 30, 35, 40, 28, 32],
#     "Occupation": ["Engineer", "Doctor", "Artist", "Teacher", "Manager", "Pilot"],
# })
#
# new_train_llm, _ = LLMGenerator(
#     gen_x_times=1.5,
#     text_generating_columns=["Name"],
#     conditional_columns=["Gender"],
#     gen_params={"batch_size": 32, "epochs": 2, "llm": "distilgpt2", "max_length": 500},
#     is_post_process=False,
# ).generate_data_pipe(train_llm, target=None, test_df=None, only_generated_data=True)
#
# print(new_train_llm)

## 5. Improving Model Performance (Breast Cancer)

In [ ]:
import sklearn.datasets
import sklearn.ensemble
import sklearn.model_selection
import sklearn.metrics
from tabgan.sampler import GANGenerator

def evaluate(clf, X_train, y_train, X_test, y_test):
    clf.fit(X_train, y_train)
    return sklearn.metrics.roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])

dataset = sklearn.datasets.load_breast_cancer()
clf = sklearn.ensemble.RandomForestClassifier(n_estimators=25, max_depth=6)
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    pd.DataFrame(dataset.data),
    pd.DataFrame(dataset.target, columns=["target"]),
    test_size=0.33, random_state=42,
)

baseline = evaluate(clf, X_train, y_train, X_test, y_test)
print(f"Baseline AUC: {baseline:.4f}")

new_train, new_target = GANGenerator(
    gen_params={"batch_size": 500, "epochs": 10}
).generate_data_pipe(X_train, y_train, X_test)
augmented = evaluate(clf, new_train, new_target, X_test, y_test)
print(f"With GAN AUC: {augmented:.4f}")

## 6. Time-Series Data Generation

In [ ]:
from tabgan.utils import get_year_mnth_dt_from_date, collect_dates
from tabgan.sampler import GANGenerator

train_ts = pd.DataFrame(np.random.randint(-10, 150, size=(100, 4)), columns=list("ABCD"))
min_date, max_date = pd.to_datetime("2019-01-01"), pd.to_datetime("2021-12-31")
d = (max_date - min_date).days + 1
train_ts["Date"] = min_date + pd.to_timedelta(np.random.randint(d, size=100), unit="d")
train_ts = get_year_mnth_dt_from_date(train_ts, "Date")

print("Before generation:")
print(train_ts.head())

new_train_ts, _ = GANGenerator(
    gen_x_times=1.1, cat_cols=["year"],
    bot_filter_quantile=0.001, top_filter_quantile=0.999,
    is_post_process=True, pregeneration_frac=2,
    gen_params={"batch_size": 500, "epochs": 10},
).generate_data_pipe(
    train_ts.drop("Date", axis=1), None, train_ts.drop("Date", axis=1)
)

new_train_ts = collect_dates(new_train_ts)
print(f"\nAfter generation ({len(new_train_ts)} rows):")
new_train_ts.head()

## 7. Quality Report

In [ ]:
from tabgan import QualityReport, GANGenerator

# Prepare data
train_q = pd.DataFrame(np.random.randint(-10, 150, size=(150, 4)), columns=list("ABCD"))
target_q = pd.DataFrame(np.random.randint(0, 2, size=(150, 1)), columns=["target"])
test_q = train_q.copy()

new_train_q, new_target_q = GANGenerator(
    gen_params={"batch_size": 500, "epochs": 10}
).generate_data_pipe(train_q, target_q, test_q)

# Build report
original_df = pd.concat([train_q, target_q], axis=1)
synthetic_df = pd.concat([new_train_q, new_target_q], axis=1)

report = QualityReport(
    original_df, synthetic_df,
    target_col="target",
).compute()

summary = report.summary()
print(f"Overall score: {summary['overall_score']}")
print(f"Mean PSI: {summary['psi']['mean']:.4f}")
print(f"ML utility ratio: {summary['ml_utility']['utility_ratio']}")

In [ ]:
# Save HTML report (download from file browser in Colab)
report.to_html("quality_report.html")
print("Report saved to quality_report.html")

In [ ]:
# Quick comparison utility
from tabgan.utils import compare_dataframes

score = compare_dataframes(original_df, synthetic_df)
print(f"Quick comparison score: {score:.4f}  (0.0 = poor, 1.0 = excellent)")

## 8. Constraints

In [ ]:
from tabgan import GANGenerator, RangeConstraint, UniqueConstraint, FormulaConstraint, RegexConstraint

# Create a dataset with meaningful columns for constraints
train_c = pd.DataFrame({
    "age": np.random.randint(18, 80, size=200),
    "salary": np.random.randint(20000, 150000, size=200),
    "score": np.random.uniform(0, 100, size=200),
    "id": range(200),
})
target_c = pd.DataFrame({"label": np.random.randint(0, 2, size=200)})

new_train_c, new_target_c = GANGenerator(
    gen_x_times=1.5,
    gen_params={"batch_size": 500, "epochs": 10},
).generate_data_pipe(
    train_c, target_c, train_c,
    constraints=[
        RangeConstraint("age", min_val=0, max_val=120),
        RangeConstraint("salary", min_val=0),
        UniqueConstraint("id"),
    ],
)

print(f"Generated {len(new_train_c)} rows with constraints applied")
print(f"Age range: [{new_train_c['age'].min():.0f}, {new_train_c['age'].max():.0f}]")
print(f"Salary min: {new_train_c['salary'].min():.0f}")
print(f"Unique IDs: {new_train_c['id'].nunique()} / {len(new_train_c)}")

In [ ]:
# Standalone ConstraintEngine
from tabgan import ConstraintEngine, RangeConstraint

noisy_df = pd.DataFrame({"price": [-5, 10, 200, -1, 50, 999]})

engine = ConstraintEngine(
    constraints=[RangeConstraint("price", min_val=0, max_val=500)],
    strategy="fix",
)
cleaned = engine.apply(noisy_df)
print("Before:", noisy_df["price"].tolist())
print("After: ", cleaned["price"].tolist())

## 9. Privacy Metrics

In [ ]:
from tabgan import PrivacyMetrics

# Reuse data from Quality Report section
pm = PrivacyMetrics(original_df, synthetic_df)
privacy = pm.summary()

print(f"Overall privacy score: {privacy['overall_privacy_score']}")
print(f"DCR mean:             {privacy['dcr']['mean']:.4f}")
print(f"NNDR mean:            {privacy['nndr']['mean']:.4f}")
print(f"Membership Inf. AUC:  {privacy['membership_inference']['auc']:.4f}  (closer to 0.5 = better)")

## 10. sklearn Pipeline Integration

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tabgan import TabGANTransformer

data = load_breast_cancer()
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    pd.DataFrame(data.data, columns=data.feature_names),
    pd.Series(data.target, name="target"),
    test_size=0.33, random_state=42,
)

# Augment training data
augmenter = TabGANTransformer(gen_x_times=1.5)
X_aug = augmenter.fit_transform(X_train_p, y_train_p)
y_aug = augmenter.get_augmented_target()

print(f"Original: {len(X_train_p)} rows -> Augmented: {len(X_aug)} rows")

# Train model on augmented data
clf = RandomForestClassifier(n_estimators=25, max_depth=6, random_state=42)
clf.fit(X_aug, y_aug)
print(f"Accuracy on test set: {accuracy_score(y_test_p, clf.predict(X_test_p)):.4f}")

In [ ]:
# Manual fit_transform + get_augmented_target pattern
from tabgan import TabGANTransformer, GANGenerator, RangeConstraint

transformer = TabGANTransformer(
    generator_class=GANGenerator,
    gen_x_times=2.0,
    gen_params={"batch_size": 500, "epochs": 10, "patience": 5},
)

X_augmented = transformer.fit_transform(X_train_p, y_train_p)
y_augmented = transformer.get_augmented_target()

print(f"Original: {X_train_p.shape[0]} rows -> Augmented: {X_augmented.shape[0]} rows")

## 11. AutoSynth

Runs all generators and picks the best one. Takes a few minutes.

In [ ]:
from tabgan import AutoSynth

df_auto = pd.concat([X_train_p, y_train_p.rename("target")], axis=1)

result = AutoSynth(df_auto, target_col="target").run()

print(result.report)
print(f"\nWinner: {result.best_name}")
result.best_data.head()

## 12. HuggingFace Hub Integration

In [ ]:
!pip install -q datasets

In [ ]:
from tabgan import synthesize_hf_dataset

result_hf = synthesize_hf_dataset("scikit-learn/iris", target_col="target")

print(f"Synthetic shape: {result_hf.synthetic_df.shape}")
print(f"Quality score:   {result_hf.quality_summary['overall_score']}")
result_hf.synthetic_df.head()

In [ ]:
# To push results back to Hub (requires authentication):
# result_hf = synthesize_hf_dataset(
#     "scikit-learn/iris",
#     target_col="target",
#     push_to_hub=True,
#     hub_repo_id="your-username/iris-synthetic",
# )

---

## Links

- [GitHub](https://github.com/Diyago/Tabular-data-generation)
- [PyPI](https://pypi.org/project/tabgan/)
- [HuggingFace Space Demo](https://huggingface.co/spaces/InsafQ/TabGAN)
- [Paper: Tabular GANs for uneven distribution](https://arxiv.org/abs/2010.00638)